In [34]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

In [35]:
def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95):
    """
    Returns (lower, upper) percentile bootstrap CI.

    Parameters
    ----------
    data : array-like
        The sample to resample from.
    stat_func : callable
        The statistic to compute on each resample (default: np.mean).
    n_boot : int
        Number of bootstrap resamples.
    ci_level : float
        Confidence level (e.g. 0.95 for a 95 % CI).
    """

    boot_stats = np.empty(n_boot)
    n = len(data)

    for i in range(n_boot):
        resample = np.random.choice(data, size=n, replace=True)
        boot_stats[i] = stat_func(resample)

    alpha = 1 - ci_level
    lower = np.percentile(boot_stats, 100 * (alpha / 2))
    upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))

    return lower, upper


In [ ]:
my_data = pd.DataFrame({
    'test_score': np.random.normal(loc = 75, scale = 10, size = 300),
})
my_data['passed'] = my_data.test_score >= 60


np.int64(284)

In [50]:
mean_ci = bootstrap_ci(my_data.test_score)
median_ci = bootstrap_ci(my_data.test_score, stat_func=np.median)
prop_ci = bootstrap_ci(my_data.passed, stat_func=np.mean)

print(f"Mean test score: {my_data.test_score.mean():.2f}\n"
      f"95% CI: [{mean_ci[0]:.2f}, {mean_ci[1]:.2f}]")

print(f"Median test score: {my_data.test_score.median():.2f}\n"
      f"Median 95% CI: [{median_ci[0]:.2f}, {median_ci[1]:.2f}]")

print(f"Proportion passed: {my_data.passed.mean():.3f}\n"
      f"Proportion 95% CI: [{prop_ci[0]:.3f}, {prop_ci[1]:.3f}]")

Mean test score: 74.94
95% CI: [73.85, 76.05]
Median test score: 75.59
Median 95% CI: [73.55, 76.75]
Proportion passed: 0.947
Proportion 95% CI: [0.920, 0.970]


In [59]:
mean_ci_st = st.norm.interval(0.95, loc=my_data.test_score.mean(), scale=st.sem(my_data.test_score))
prop_ci_st = st.t.interval(0.95, df=len(my_data)-1, loc=my_data.passed.mean(), scale=st.sem(my_data.passed))

print(f"Mean 95% CI manually: [{mean_ci[0]:.2f}, {mean_ci[1]:.2f}]")
print(f"Mean 95% CI using scipy: [{mean_ci_st[0]:.2f}, {mean_ci_st[1]:.2f}]")

print(f"Proportion 95% CI manually: [{prop_ci[0]:.3f}, {prop_ci[1]:.3f}]")
print(f"Proportion 95% CI using scipy: [{prop_ci_st[0]:.3f}, {prop_ci_st[1]:.3f}]") 

Mean 95% CI manually: [73.85, 76.05]
Mean 95% CI using scipy: [73.83, 76.06]
Proportion 95% CI manually: [0.920, 0.970]
Proportion 95% CI using scipy: [0.921, 0.972]


The confidence intervals for both the mean and the proportion are very similar whether calculated manually (bootstrap) or using SciPy:

Mean test score: 
- Manual CI [73.85, 76.05]
- SciPy CI [73.83, 76.06]

Proportion passed: 
- Manual CI [0.920, 0.970]
- SciPy CI [0.921, 0.972]

This tells us that for large samples, both approaches tend to agree closely. The small differences we see are just due to minor rounding differences or the exact calculation method (like using a t-distribution versus a normal approximation).

Where these methods can diverge is when the data is skewed, the sample size is small, or the statistic doesn’t have a simple formula. In particular, the median is a good example. There’s no straightforward formula for its standard error, and its distribution can be asymmetric. Here, the bootstrap is very useful because it estimates the confidence interval empirically by resampling, without assuming normality.
